In [1]:
import importlib
import NeuralNetwork
import funcs
import plots

importlib.reload(NeuralNetwork)
importlib.reload(funcs)
importlib.reload(plots)
from NeuralNetwork import NeuralNetwork

import torch
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, random_split
from collections import defaultdict
import numpy as np
from tqdm.auto import tqdm
import copy
import os

In [ ]:
%run shared.py

## Activation analysis and Neuron Clustering

In [9]:
if os.path.exists("pruned_model.pth"):
    final_model = torch.load("pruned_model.pth", weights_only=False)
    final_model.eval()
else:
    print("No final model saved on disk")

We create a new data loader that does not shuffle the data so our instances match up with our data

In [ ]:
analysis_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=False)
analysis_model = copy.deepcopy(final_model)
analysis_model.eval()

In [ ]:
layer_data_analysis = analysis_model.get_layer_data(analysis_loader)

In [ ]:
for key, value in layer_data_analysis.items():
    print(f"{key} has shape: {value['post_activation'].shape}")

In [ ]:
hidden_layers = [k for k in layer_data_analysis.keys() if 'layer_' in k]

all_neuron_activations = []
for layer_name in hidden_layers:
    acts = layer_data_analysis[layer_name]['post_activation']
    all_neuron_activations.append(acts)

all_neuron_activations = torch.cat(all_neuron_activations, dim=1)

In [ ]:
layer_mapping = []
start_idx = 0
for layer_name in hidden_layers:
    n_neurons = layer_data_analysis[layer_name]['post_activation'].shape[1]
    end_idx = start_idx + n_neurons
    layer_mapping.append((layer_name, start_idx, end_idx))
    start_idx = end_idx

In [ ]:
labels = []
images = []
for X, y in tqdm(analysis_loader):
    labels.append(y)
    images.append(X)
labels = torch.cat(labels, dim=0)
images = torch.cat(images, dim=0)

## Clustering

In [ ]:
from sklearn.decomposition import NMF

NMF_SUBSAMPLE = 15000
activations_np = all_neuron_activations.numpy()  # [N_samples, N_neurons]

if activations_np.shape[0] > NMF_SUBSAMPLE:
    rng = np.random.default_rng(42)
    idx = rng.choice(activations_np.shape[0], size=NMF_SUBSAMPLE, replace=False)
    activations_fit = activations_np[idx]
else:
    activations_fit = activations_np

nmf = NMF(n_components=N_CLUSTERS, random_state=42, max_iter=500)
W = nmf.fit_transform(activations_fit)  # [N_samples_fit, N_CLUSTERS]
H = nmf.components_                     # [N_CLUSTERS, N_neurons]
neuron_assignments = H.argmax(axis=0) + 1  # 1-indexed cluster IDs
clusters = neuron_assignments

print(f"Total neurons clustered: {activations_np.shape[1]}")
print(f"Clusters assigned: {np.unique(clusters)}")

# Build cluster_map: {cluster_id: [neuron_indices]}
cluster_map = defaultdict(list)
for neuron_idx, cluster_id in enumerate(clusters):
    cluster_map[cluster_id].append(neuron_idx)

## Cluster selectivity analysis

In [ ]:
selectivity_results = funcs.compute_cluster_selectivity(cluster_map, all_neuron_activations, labels)

In [ ]:
funcs.plot_cluster_activation_heatmap(selectivity_results)

## Cluster ablation

In [ ]:
cluster_results = {}
for cluster_id, neuron_indices in cluster_map.items():
    per_class_acc = funcs.cluster_criticality_per_class(
        analysis_model,
        neuron_indices,
        layer_mapping,
        val_loader,
        cluster_id,
        device=device
    )
    cluster_results[cluster_id] = per_class_acc

In [ ]:
print(f"Accuracy: {analysis_model.accuracy(val_loader):.2f}")

In [ ]:
cluster_class_changes = plots.plot_cluster_accuracy_bars(cluster_results, target_labels=list(range(10)))

## Prototype and difference map plots

In [ ]:
# Compute prototypes and difference maps for all clusters
all_prototypes = funcs.compute_prototypes_all_clusters(
    cluster_map=cluster_map,
    all_activations=all_neuron_activations,
    images=images,
    top_frac=0.1,           # fraction of top-activating samples per cluster
    use_global_mean=True     # normalize diff map relative to global mean
)

In [ ]:
plots.plot_cluster_prototypes_and_diff_all(all_prototypes)

## cluster extraction